In [ ]:
import nltk
import re

In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 73.0 MB/s eta 0:00:00


In [ ]:
nltk.download('gutenberg')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

from nltk.corpus import gutenberg

[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Unzipping corpora/gutenberg.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


In [ ]:
raw_text = gutenberg.raw('shakespeare-hamlet.txt')
print("Sample raw text:\n", raw_text[:300], "\n")

Sample raw text:
 [The Tragedie of Hamlet by William Shakespeare 1599]


Actus Primus. Scoena Prima.

Enter Barnardo and Francisco two Centinels.

  Barnardo. Who's there?
  Fran. Nay answer me: Stand & vnfold
your selfe

   Bar. Long liue the King

   Fran. Barnardo?
  Bar. He

   Fran. You come most carefully vpon  



In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)   # keep only letters
    text = re.sub(r'\s+', ' ', text).strip()
    return text

clean = clean_text(raw_text)

In [ ]:
from nltk.tokenize import sent_tokenize, word_tokenize

sentences = sent_tokenize(raw_text)             # sentence split (on raw text, before cleaning)
tokenized_sentences = [word_tokenize(clean_text(s)) for s in sentences]
tokenized_sentences = [s for s in tokenized_sentences if len(s) > 0]

print("Number of sentences:", len(tokenized_sentences))
print("Sample tokenized sentence:", tokenized_sentences[0], "\n")

Number of sentences: 2355
Sample tokenized sentence: ['the', 'tragedie', 'of', 'hamlet', 'by', 'william', 'shakespeare', 'actus', 'primus'] 



In [ ]:
# =====================================================
# 3. SUBWORD TOKENIZATION: BPE and WordPiece
# =====================================================
from tokenizers import Tokenizer
from tokenizers.models import BPE, WordPiece
from tokenizers.trainers import BpeTrainer, WordPieceTrainer
from tokenizers.pre_tokenizers import Whitespace

# Write cleaned text to a temp file (tokenizers library trains from files)
with open("shakespeare_clean.txt", "w") as f:
    f.write(clean)

In [ ]:
# ---- BPE ----
bpe_tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
bpe_tokenizer.pre_tokenizer = Whitespace()
bpe_trainer = BpeTrainer(vocab_size=2000, special_tokens=["[UNK]"])
bpe_tokenizer.train(["shakespeare_clean.txt"], bpe_trainer)

bpe_output = bpe_tokenizer.encode("To be or not to be")
print("BPE tokens:", bpe_output.tokens)

BPE tokens: ['[UNK]', 'o', 'be', 'or', 'not', 'to', 'be']


In [ ]:
# ---- WordPiece ----
wp_tokenizer = Tokenizer(WordPiece(unk_token="[UNK]"))
wp_tokenizer.pre_tokenizer = Whitespace()
wp_trainer = WordPieceTrainer(vocab_size=2000, special_tokens=["[UNK]"])
wp_tokenizer.train(["shakespeare_clean.txt"], wp_trainer)

wp_output = wp_tokenizer.encode("To be or not to be")
print("WordPiece tokens:", wp_output.tokens, "\n")

WordPiece tokens: ['[UNK]', 'be', 'or', 'not', 'to', 'be'] 



In [ ]:
# =====================================================
# 4. POS TAGGING
# =====================================================
from nltk import pos_tag

sample_sentence = word_tokenize(sentences[5])   # use an original (uncleaned) sentence
pos_tags = pos_tag(sample_sentence)
print("POS tags for a sample sentence:")
print(pos_tags, "\n")

POS tags for a sample sentence:
[('Fran', 'NNP'), ('.', '.')] 



In [ ]:
# =====================================================
# 5. WORD EMBEDDINGS
# =====================================================
from gensim.models import Word2Vec, FastText

# ---- Word2Vec ----
w2v_model = Word2Vec(sentences=tokenized_sentences, vector_size=100, window=5, min_count=2, sg=1)
print("Word2Vec vector for 'king':\n", w2v_model.wv['king'][:10], "...")
print("Word2Vec most similar to 'king':", w2v_model.wv.most_similar('king', topn=5), "\n")

Word2Vec vector for 'king':
 [-0.17476608  0.15366817  0.00096417  0.00855197 -0.01805222 -0.32573286
  0.01516702  0.38810912 -0.16686182 -0.09719012] ...
Word2Vec most similar to 'king': [('graue', 0.9978011250495911), ('within', 0.9976010322570801), ('custome', 0.9975982308387756), ('tongue', 0.997511625289917), ('eye', 0.9974671006202698)] 



In [ ]:

# ---- FastText ----
ft_model = FastText(sentences=tokenized_sentences, vector_size=100, window=5, min_count=2, sg=1)
print("FastText vector for 'king':\n", ft_model.wv['king'][:10], "...")
print("FastText most similar to 'king':", ft_model.wv.most_similar('king', topn=5), "\n")

FastText vector for 'king':
 [ 0.1242925   0.18141769 -0.29582387 -0.12057602 -0.09923304 -0.03494843
  0.06535839  0.54846233 -0.07458469  0.12959681] ...
FastText most similar to 'king': [('kings', 0.9999174475669861), ('making', 0.9999034404754639), ('kinde', 0.9998917579650879), ('working', 0.999891459941864), ('likes', 0.9998716711997986)] 



In [ ]:
 #---- GloVe ----
# Training GloVe from scratch needs a separate co-occurrence + optimization step
# (not in gensim). Easiest simple approach: load pretrained GloVe vectors
# and just look up your corpus words in them.
import gensim.downloader as api

glove_model = api.load("glove-wiki-gigaword-100")  # downloads once, ~128MB
if 'king' in glove_model:
    print("GloVe vector for 'king':\n", glove_model['king'][:10], "...")
    print("GloVe most similar to 'king':", glove_model.most_similar('king', topn=5))

[==================================================] 100.0% 128.1/128.1MB downloaded
GloVe vector for 'king':
 [-0.32307 -0.87616  0.21977  0.25268  0.22976  0.7388  -0.37954 -0.35307
 -0.84369 -1.1113 ] ...
GloVe most similar to 'king': [('prince', 0.7682328820228577), ('queen', 0.7507690787315369), ('son', 0.7020888328552246), ('brother', 0.6985775232315063), ('monarch', 0.6977890729904175)]
